In [1]:
import pandas as pd

df = pd.read_csv('data/thai_wikipron_5-4-2026.csv')
df

,writing,phonetic,count
0,ก,kɔː˧,2
1,ก,kɔː˧.kaj˨˩,2
2,ก.,kɔː˧,1
3,ก.ค.,kɔː˧.kʰɔː˧,1
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1
...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2
18307,ไฮ้,haj˦˥,1
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1


In [2]:
import re

def normalize_phonetic(phonetic: str) -> str:
    phonetic = (
        phonetic
        .replace('p̚', 'p')
        .replace('t̚', 't')
        .replace('k̚', 'k')
        .replace('a̯', 'ə')
        .replace('˥˩', '˦˩')
        .replace('˩˩˦', '˨˥')
    )

    SHORT_VOWELS = ['a', 'i', 'ɯ', 'u', 'e', 'ɤ', 'o', 'ɛ', 'ɔ']

    pattern = (
        '(' + '|'.join(map(re.escape, SHORT_VOWELS)) + ')'
        r'([˥˦˧˨˩]+)(?=\.|$)'
    )

    phonetic = re.sub(pattern, r'\1ʔ\2', phonetic)

    phonetic = re.sub(r'[.…]+', '.', phonetic)

    phonetic = re.sub(r'^\.|\.$', '', phonetic)

    return phonetic

print(normalize_phonetic('….daj˧.….nɯŋ˨˩'))

daj˧.nɯŋ˨˩


In [3]:
df['normalized_phonetic'] = df['phonetic'].apply(normalize_phonetic)
df

,writing,phonetic,count,normalized_phonetic
0,ก,kɔː˧,2,kɔː˧
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩
2,ก.,kɔː˧,1,kɔː˧
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧
...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩


In [4]:
import thai_gpa

def evaluate(text: str, ipa: str) -> tuple:
    result = thai_gpa.align(text, ipa)
    reconstructed_text = ''.join(s.reconstruct_text() for s in result)
    reconstructed_ipa = '.'.join(s.get_ipa(is_reduplicated=s.is_reduplicated) for s in result)
    return reconstructed_text, reconstructed_ipa

print(evaluate('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩'))

('การขัดกันของผลประโยชน์', 'kaːn˧.kʰat˨˩.kan˧.kʰɔːŋ˨˥.pʰon˨˥.praʔ˨˩.joːt˨˩')


In [5]:
from tqdm.auto import tqdm
tqdm.pandas()

def apply_evaluate(row):
    try:
        reconstructed_text, phonetic_answer = evaluate(row['writing'], row['normalized_phonetic'])
    except Exception as e:
        reconstructed_text, phonetic_answer = None, f'ERROR: {e}'
    return pd.Series([reconstructed_text, phonetic_answer])

df[['reconstructed_text', 'phonetic_answer']] = df.progress_apply(apply_evaluate, axis=1)
df.to_csv('data/test.csv', index=False)
df

C:\Users\pawi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 91%|█████████ | 16679/18310 [05:19<00:19, 83.97it/s] c:\Storage\repos\thai_grapheme_sandbox\thai_ipa.py:109: UserWarning: Warning: No explicit glottal stop at "e"
  warnings.warn(f'Warning: No explicit glottal stop at "{original}"')
100%|██████████| 18310/18310 [05:41<00:00, 53.56it/s] 


,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,ก,kɔːʔ˧
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,None,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,ก.,kɔːʔ˧
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,ก.ค.,kɔːk˧.kʰɔːʔ˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,ก.ท.ม.,kɔːʔ˧.tʰɔːm˧.mɔːʔ˧
...,...,...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥,ไฮ้,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,None,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


# Analyze

In [1]:
import pandas as pd

df = pd.read_csv('data/test.csv')
df

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
0,ก,kɔː˧,2,kɔː˧,ก,kɔːʔ˧
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,NaN,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
2,ก.,kɔː˧,1,kɔː˧,ก.,kɔːʔ˧
3,ก.ค.,kɔː˧.kʰɔː˧,1,kɔː˧.kʰɔː˧,ก.ค.,kɔːk˧.kʰɔːʔ˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,1,kɔː˧.tʰɔː˧.mɔː˧,ก.ท.ม.,kɔːʔ˧.tʰɔːm˧.mɔːʔ˧
...,...,...,...,...,...,...
18305,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧,2,haj˧.droː˧.t͡ɕeːn˧,ไฮโดรเจน,haj˧.droː˧.t͡ɕeːn˧
18306,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˥˩,2,haj˧.droː˧.t͡ɕen˦˩,ไฮโดรเจน,haj˧.droː˧.t͡ɕen˦˩
18307,ไฮ้,haj˦˥,1,haj˦˥,ไฮ้,haj˦˥
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,NaN,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [2]:
mask = ~df["phonetic_answer"].str.startswith("ERROR:")
mismatches = df.loc[
    mask & (df["normalized_phonetic"] != df["phonetic_answer"]),
    ["writing", "normalized_phonetic", "phonetic_answer"]
]

mismatches

,writing,normalized_phonetic,phonetic_answer
0,ก,kɔː˧,kɔːʔ˧
2,ก.,kɔː˧,kɔːʔ˧
3,ก.ค.,kɔː˧.kʰɔː˧,kɔːk˧.kʰɔːʔ˧
4,ก.ท.ม.,kɔː˧.tʰɔː˧.mɔː˧,kɔːʔ˧.tʰɔːm˧.mɔːʔ˧
5,ก.พ.,kɔː˧.pʰɔː˧,kɔːp˧.pʰɔːʔ˧
...,...,...,...
16971,แล้วก็,lɛːw˦˥.kɔː˦˩,lɛːw˦˥.kɔːʔ˦˩
17333,โทร,tʰoː˧,tʰoːn˧
17467,โฟโตชอป,foː˧.toː˦˩.t͡ɕʰɔp˨˩,foː˧.toːt˦˩.t͡ɕʰɔp˨˩
18016,ไทแรนโนซอรัส,tʰaj˧.rɛː˧.noː˧.sɔː˧.ras˦˥,tʰaj˧.rɛːn˧.noː˧.sɔː˧.ras˦˥


In [3]:
errors = df[df["phonetic_answer"].str.startswith("ERROR:")]
errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,NaN,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
374,กราฟ,kraːf˦˥,2,kraːf˦˥,NaN,ERROR: argument of type 'NoneType' is not iter...
568,กษัตรา,kra˨˩.sat̚˨˩.traː˧,1,kraʔ˨˩.sat˨˩.traː˧,NaN,ERROR: Could not align 'กษัตรา' with 'kraʔ˨˩.s...
571,กษัตรีย์,ka˨˩.sat̚˨˩.triː˧,1,kaʔ˨˩.sat˨˩.triː˧,NaN,ERROR: Could not align 'กษัตรีย์' with 'kaʔ˨˩....
611,กอริลลา,kɔː˧.ril˧.laː˥˩,2,kɔː˧.ril˧.laː˦˩,NaN,ERROR: argument of type 'NoneType' is not iter...
...,...,...,...,...,...,...
18262,ไอซ์แลนด์,ʔajs˦˥.lɛːn˧,1,ʔajs˦˥.lɛːn˧,NaN,ERROR: argument of type 'NoneType' is not iter...
18263,ไอดอล,ʔaj˧.dɔl˥˩,2,ʔaj˧.dɔl˦˩,NaN,ERROR: argument of type 'NoneType' is not iter...
18278,ไอศครีม,ʔajs˧.kʰriːm˧,2,ʔajs˧.kʰriːm˧,NaN,ERROR: argument of type 'NoneType' is not iter...
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,NaN,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [4]:
errors.sample(10)

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
1463,การ์ฟิลด์,kaː˧.fil˧,2,kaː˧.fil˧,NaN,ERROR: argument of type 'NoneType' is not iter...
17717,โอลีฟ,ʔoː˧.liːf˦˥,1,ʔoː˧.liːf˦˥,NaN,ERROR: argument of type 'NoneType' is not iter...
6504,บล็อกเกอร์,blɔk̚˦˥.kɤː˥˩,1,blɔk˦˥.kɤː˦˩,NaN,ERROR: Could not align 'บล็อกเกอร์' with 'blɔk...
17058,แองกวิลลา,ʔɛŋ˧.kwil˧.laː˥˩,1,ʔɛŋ˧.kwil˧.laː˦˩,NaN,ERROR: argument of type 'NoneType' is not iter...
4198,ฑาก,daː˧.ka˨˩.,1,daː˧.kaʔ˨˩,NaN,ERROR: Could not align 'ฑาก' with 'daː˧.kaʔ˨˩'
4196,ฑ,tʰɔː˧.mon˧.tʰoː˧,2,tʰɔː˧.mon˧.tʰoː˧,NaN,ERROR: Could not align 'ฑ' with 'tʰɔː˧.mon˧.tʰ...
14214,เกล็ดเลือดน้อย,kret̚˨˩.lɯa̯t̚˥˩.nɔːj˦˥,1,kret˨˩.lɯət˦˩.nɔːj˦˥,NaN,ERROR: Could not align 'เกล็ดเลือดน้อย' with '...
4343,ดิจิทัล,di˨˩.t͡ɕi˨˩.tʰal˥˩,2,diʔ˨˩.t͡ɕiʔ˨˩.tʰal˦˩,NaN,ERROR: argument of type 'NoneType' is not iter...
17502,โมเดล,moː˧.del˧,1,moː˧.del˧,NaN,ERROR: argument of type 'NoneType' is not iter...
9610,ร,rɔː˧.rɯa̯˧,3,rɔː˧.rɯə˧,NaN,ERROR: Could not align 'ร' with 'rɔː˧.rɯə˧'


In [5]:
mask = ~errors["normalized_phonetic"].str.contains(
    r"[lsf][˥˦˧˨˩]", regex=True, na=False
)
filtered_errors = errors[mask]
filtered_errors

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
1,ก,kɔː˧.kaj˨˩,2,kɔː˧.kaj˨˩,NaN,ERROR: Could not align 'ก' with 'kɔː˧.kaj˨˩'
568,กษัตรา,kra˨˩.sat̚˨˩.traː˧,1,kraʔ˨˩.sat˨˩.traː˧,NaN,ERROR: Could not align 'กษัตรา' with 'kraʔ˨˩.s...
571,กษัตรีย์,ka˨˩.sat̚˨˩.triː˧,1,kaʔ˨˩.sat˨˩.triː˧,NaN,ERROR: Could not align 'กษัตรีย์' with 'kaʔ˨˩....
755,กาฐมาณฑุ,kaːt̚˨˩.maːn˧.duʔ˨˩,1,kaːt˨˩.maːn˧.duʔ˨˩,NaN,ERROR: Could not align 'กาฐมาณฑุ' with 'kaːt˨˩...
828,การข่มขืนกระทำชำเรา,kaːn˧.t͡ɕʰom˥˩.kʰɯːn˩˩˦.kra˨˩.tʰam˧.t͡ɕʰam˧.raw˧,1,kaːn˧.t͡ɕʰom˦˩.kʰɯːn˨˥.kraʔ˨˩.tʰam˧.t͡ɕʰam˧.raw˧,NaN,ERROR: Could not align 'การข่มขืนกระทำชำเรา' w...
...,...,...,...,...,...,...
17403,โปรดเกล้าฯ,proːt̚˨˩.klaːw˥˩,1,proːt˨˩.klaːw˦˩,NaN,ERROR: Could not align 'โปรดเกล้าฯ' with 'proː...
17701,โอฑิศา,ʔoː˧.di˨˩.saː˩˩˦,1,ʔoː˧.diʔ˨˩.saː˨˥,NaN,ERROR: Could not align 'โอฑิศา' with 'ʔoː˧.diʔ...
18009,ไทร,saj˧,1,saj˧,NaN,ERROR: Could not align 'ไทร' with 'saj˧'
18308,…ใด…หนึ่ง,….daj˧.….nɯŋ˨˩,1,daj˧.nɯŋ˨˩,NaN,ERROR: Could not align '…ใด…หนึ่ง' with 'daj˧....


In [6]:
filtered_errors.sample(10)

,writing,phonetic,count,normalized_phonetic,reconstructed_text,phonetic_answer
18009,ไทร,saj˧,1,saj˧,NaN,ERROR: Could not align 'ไทร' with 'saj˧'
8499,ฟลีโรเวียม,fliː˧.roː˧.wia̯m˥˩,1,fliː˧.roː˧.wiəm˦˩,NaN,ERROR: Could not align 'ฟลีโรเวียม' with 'fliː...
9939,ราชบัณฑิต,raːt̚˥˩.t͡ɕʰa˦˥.ban˧.dit̚˨˩,1,raːt˦˩.t͡ɕʰaʔ˦˥.ban˧.dit˨˩,NaN,ERROR: Could not align 'ราชบัณฑิต' with 'raːt˦...
5005,ถ,tʰɔː˩˩˦.tʰuŋ˩˩˦,2,tʰɔː˨˥.tʰuŋ˨˥,NaN,ERROR: Could not align 'ถ' with 'tʰɔː˨˥.tʰuŋ˨˥'
8779,มณฑป,mon˧.dop̚˨˩,1,mon˧.dop˨˩,NaN,ERROR: Could not align 'มณฑป' with 'mon˧.dop˨˩'
2027,ข่าง,xaːŋ˨˩,3,xaːŋ˨˩,NaN,ERROR: Could not align 'ข่าง' with 'xaːŋ˨˩'
14073,ฮ,hɔː˧.nok̚˦˥.huːk̚˥˩,2,hɔː˧.nok˦˥.huːk˦˩,NaN,ERROR: Could not align 'ฮ' with 'hɔː˧.nok˦˥.hu...
6547,บัควีท,bak̚˦˥.hwiːt̚˨˩,2,bak˦˥.hwiːt˨˩,NaN,ERROR: Could not align 'บัควีท' with 'bak˦˥.hw...
5010,ถถถ,haː˥˩.haː˥˩.haː˥˩,2,haː˦˩.haː˦˩.haː˦˩,NaN,ERROR: Could not align 'ถถถ' with 'haː˦˩.haː˦˩...
6575,บัณฑิต,ban˧.dit̚˨˩,1,ban˧.dit˨˩,NaN,ERROR: Could not align 'บัณฑิต' with 'ban˧.dit˨˩'
